In [ ]:
import os
import re
import random
from collections import Counter
from dataclasses import dataclass, replace
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip /content/drive/MyDrive/dataset.zip -d Dataset_BUSI_with_GT

In [ ]:
# -----------------------------
# 0) Reproducibility utilities
# -----------------------------
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Keep runs reproducible. Set benchmark=True after debugging if you prefer speed.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ---------------------------------
# 1) Configuration (single source)
# ---------------------------------
@dataclass
class Config:
    data_root: str  # path to BUSI root that contains benign/malignant/normal
    img_size: int = 224
    batch_size: int = 32
    num_workers: int = 4
    seed: int = 42

    # Splits are done by original image id, not by file, to prevent augmented leakage.
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    # Training hyperparams
    model_name: str = "convnext_tiny"
    pretrained: bool = True
    epochs: int = 35
    freeze_epochs: int = 3
    lr: float = 8e-5
    min_lr: float = 1e-6
    weight_decay: float = 1e-4
    label_smoothing: float = 0.02
    class_weight_power: float = 0.35
    focal_gamma: float = 0.0
    mixup_alpha: float = 0.0
    mixup_prob: float = 0.0
    sampler_class_boost: Optional[List[float]] = None
    patience: int = 8

    # Pre-augmented *_augN files are allowed only in train. Val/test stay original-only.
    # For transfer learning, online augmentation usually generalizes better than replaying a fixed augmented set.
    use_preaugmented_train: bool = False
    use_online_train_augmentation: bool = True
    use_weighted_sampler: bool = True
    train_epoch_multiplier: float = 1.0

    # BUSI includes ground-truth masks. Exactly one mask-assisted input mode may be enabled.
    # Both modes require a lesion mask at inference and must be reported as mask-assisted.
    use_mask_roi_crop: bool = True
    use_mask_geometry_channels: bool = False
    roi_margin: float = 0.15

    apply_basic_transforms: bool = True


# -----------------------------
# 2) Robust image file listing
# -----------------------------
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
AUGMENTED_RE = re.compile(r"_aug\d+(?=\.[^.]+$)", re.IGNORECASE)


def is_mask_file(path: str) -> bool:
    name = os.path.basename(path).lower()
    return ("_mask" in name) or name.endswith("mask.png")


def is_augmented_file(path: str) -> bool:
    return AUGMENTED_RE.search(os.path.basename(path)) is not None


def original_image_key(path: str) -> str:
    """Stable key that maps original and *_augN variants to the same image id."""
    p = Path(path)
    stem = AUGMENTED_RE.sub("", p.name)
    return str(p.with_name(stem))


def is_image_file(path: str) -> bool:
    ext_ok = os.path.splitext(path.lower())[1] in IMG_EXTS
    return ext_ok and not is_mask_file(path)


def mask_paths_for_image(path: str) -> List[str]:
    p = Path(path)
    base = AUGMENTED_RE.sub("", p.stem)
    candidates = []
    for ext in IMG_EXTS:
        candidates.extend(p.parent.glob(f"{base}_mask*{ext}"))
    return sorted(str(x) for x in candidates if x.is_file())


def crop_image_to_mask_roi(img: Image.Image, mask_paths: List[str], margin: float = 0.15) -> Image.Image:
    if not mask_paths:
        return img

    union = None
    for mp in mask_paths:
        try:
            with Image.open(mp) as m:
                arr = np.array(m.convert("L")) > 0
        except Exception:
            continue
        union = arr if union is None else (union | arr)

    if union is None or not union.any():
        return img

    ys, xs = np.where(union)
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    w, h = img.size
    pad = int(max(x1 - x0, y1 - y0) * margin)
    x0 = max(0, x0 - pad)
    y0 = max(0, y0 - pad)
    x1 = min(w, x1 + pad)
    y1 = min(h, y1 + pad)

    if x1 <= x0 or y1 <= y0:
        return img
    return img.crop((x0, y0, x1, y1))


def list_images_in_class_folder(class_dir: str) -> List[str]:
    paths = []
    for root, _, files in os.walk(class_dir):
        for fn in files:
            full = os.path.join(root, fn)
            if is_image_file(full):
                paths.append(full)
    return sorted(paths)


# --------------------------------------------
# 3) Dataset that does NOT assume train/val split
# --------------------------------------------
class BUSIClassificationDataset(Dataset):
    def __init__(
        self,
        samples: List[Tuple[str, int]],
        transform: Optional[transforms.Compose] = None,
        class_names: Optional[List[str]] = None,
        use_mask_roi_crop: bool = False,
        use_mask_geometry_channels: bool = False,
        roi_margin: float = 0.15,
    ) -> None:
        self.samples = samples
        self.transform = transform
        self.class_names = class_names or ["benign", "malignant", "normal"]
        self.use_mask_roi_crop = use_mask_roi_crop
        self.use_mask_geometry_channels = use_mask_geometry_channels
        self.roi_margin = roi_margin

        if len(self.samples) == 0:
            raise ValueError("Dataset has 0 samples. Check data_root and folder names.")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        path, label = self.samples[idx]
        with Image.open(path) as opened:
            gray = np.asarray(opened.convert("L"), dtype=np.uint8).copy()

        if self.use_mask_geometry_channels:
            union = np.zeros_like(gray, dtype=np.uint8)
            for mask_path in mask_paths_for_image(path):
                try:
                    with Image.open(mask_path) as opened_mask:
                        candidate = np.asarray(opened_mask.convert("L"), dtype=np.uint8)
                except Exception:
                    continue
                union = np.maximum(union, (candidate > 0).astype(np.uint8) * 255)
            masked = np.where(union > 0, gray, 0).astype(np.uint8)
            # R=context, G=binary lesion geometry, B=lesion-only texture.
            img = Image.merge("RGB", (Image.fromarray(gray), Image.fromarray(union), Image.fromarray(masked)))
        else:
            img = Image.fromarray(gray).convert("RGB")
            if self.use_mask_roi_crop:
                img = crop_image_to_mask_roi(img, mask_paths_for_image(path), margin=self.roi_margin)

        if self.transform is not None:
            img = self.transform(img)

        return {
            "image": img,
            "label": torch.tensor(label, dtype=torch.long),
            "path": path,
        }


# --------------------------------------------
# 4) Build grouped samples + split deterministically
# --------------------------------------------
def build_image_groups_from_root(data_root: str) -> Tuple[List[Dict[str, object]], List[str]]:
    """
    Groups original BUSI images with their *_augN files. Splitting uses one group as one unit,
    so augmented variants cannot leak into validation/test.
    """
    class_names = ["benign", "malignant", "normal"]
    class_to_idx = {c: i for i, c in enumerate(class_names)}

    missing = [c for c in class_names if not os.path.isdir(os.path.join(data_root, c))]
    if missing:
        raise FileNotFoundError(f"Missing class folders in {data_root}: {missing}. Expected: {class_names}")

    groups_by_key: Dict[Tuple[int, str], Dict[str, object]] = {}
    for c in class_names:
        cdir = os.path.join(data_root, c)
        paths = list_images_in_class_folder(cdir)
        if len(paths) == 0:
            raise FileNotFoundError(f"No images found in: {cdir}")

        y = class_to_idx[c]
        for p in paths:
            key = original_image_key(p)
            group = groups_by_key.setdefault((y, key), {"label": y, "original": None, "augmented": []})
            if is_augmented_file(p):
                group["augmented"].append(p)
            else:
                group["original"] = p

    groups = []
    for group in groups_by_key.values():
        if group["original"] is None:
            # Fallback for datasets that contain only augmented-like filenames.
            aug_paths = sorted(group["augmented"])
            group["original"] = aug_paths[0]
            group["augmented"] = aug_paths[1:]
        group["augmented"] = sorted(group["augmented"])
        groups.append(group)

    return groups, class_names


def groups_to_samples(
    groups: List[Dict[str, object]],
    include_augmented: bool,
    allowed_labels: Optional[set] = None,
    relabel_map: Optional[Dict[int, int]] = None,
) -> List[Tuple[str, int]]:
    samples: List[Tuple[str, int]] = []
    for group in groups:
        original_y = int(group["label"])
        if allowed_labels is not None and original_y not in allowed_labels:
            continue
        y = relabel_map.get(original_y, original_y) if relabel_map is not None else original_y
        samples.append((str(group["original"]), y))
        if include_augmented:
            samples.extend((str(p), y) for p in group["augmented"])
    return samples


def stratified_group_split(
    groups: List[Dict[str, object]],
    val_ratio: float,
    test_ratio: float,
    seed: int,
) -> Tuple[List[Dict[str, object]], List[Dict[str, object]], List[Dict[str, object]]]:
    if val_ratio + test_ratio >= 1.0:
        raise ValueError("val_ratio + test_ratio must be < 1.0")

    rng = np.random.default_rng(seed)
    by_label: Dict[int, List[Dict[str, object]]] = {}
    for group in groups:
        by_label.setdefault(int(group["label"]), []).append(group)

    train, val, test = [], [], []
    for y, items in by_label.items():
        items = items.copy()
        rng.shuffle(items)

        n = len(items)
        n_test = max(1, int(round(n * test_ratio)))
        n_val = max(1, int(round(n * val_ratio)))
        if n - n_test - n_val <= 0:
            raise ValueError(f"After split, class {y} has 0 train samples. Reduce ratios.")

        test.extend(items[:n_test])
        val.extend(items[n_test:n_test + n_val])
        train.extend(items[n_test + n_val:])

    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)
    return train, val, test


# --------------------------------------------
# 5) Transforms
# --------------------------------------------
def get_train_transforms(img_size: int, use_online_augmentation: bool, preserve_full_frame: bool = False) -> transforms.Compose:
    if use_online_augmentation and not preserve_full_frame:
        ops = [
            transforms.RandomResizedCrop(
                img_size,
                scale=(0.92, 1.0),
                ratio=(0.95, 1.05),
                interpolation=transforms.InterpolationMode.BILINEAR,
            )
        ]
    else:
        ops = [transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.BILINEAR)]
    if use_online_augmentation:
        ops.extend([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=5 if preserve_full_frame else 7),
            transforms.RandomAffine(degrees=0, translate=(0.02, 0.02), scale=(0.98, 1.02)),
        ])
        if not preserve_full_frame:
            ops.append(transforms.ColorJitter(brightness=0.08, contrast=0.10))
    ops.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return transforms.Compose(ops)


def get_eval_transforms(img_size: int) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def make_weighted_sampler(
    samples: List[Tuple[str, int]],
    num_classes: int,
    seed: int,
    class_boost: Optional[List[float]] = None,
    epoch_size_multiplier: float = 1.0,
) -> WeightedRandomSampler:
    counts = Counter(y for _, y in samples)
    weights = []
    for _, y in samples:
        boost = 1.0
        if class_boost is not None and int(y) < len(class_boost):
            boost = float(class_boost[int(y)])
        weights.append(boost / counts[y])
    generator = torch.Generator()
    generator.manual_seed(seed)
    num_samples = max(len(weights), int(round(len(weights) * float(epoch_size_multiplier))))
    return WeightedRandomSampler(weights=weights, num_samples=num_samples, replacement=True, generator=generator)


# --------------------------------------------
# 6) Loader builder
# --------------------------------------------
def make_loaders_from_group_splits(
    cfg: Config,
    train_g: List[Dict[str, object]],
    val_g: List[Dict[str, object]],
    test_g: List[Dict[str, object]],
    class_names: List[str],
    allowed_labels: Optional[set] = None,
    relabel_map: Optional[Dict[int, int]] = None,
    display_name: str = "3-class",
):
    train_s = groups_to_samples(
        train_g,
        include_augmented=cfg.use_preaugmented_train,
        allowed_labels=allowed_labels,
        relabel_map=relabel_map,
    )
    val_s = groups_to_samples(val_g, include_augmented=False, allowed_labels=allowed_labels, relabel_map=relabel_map)
    test_s = groups_to_samples(test_g, include_augmented=False, allowed_labels=allowed_labels, relabel_map=relabel_map)

    if cfg.apply_basic_transforms:
        train_transform = get_train_transforms(
            cfg.img_size, cfg.use_online_train_augmentation, preserve_full_frame=cfg.use_mask_geometry_channels
        )
        eval_transform = get_eval_transforms(cfg.img_size)
    else:
        train_transform = eval_transform = None

    train_ds = BUSIClassificationDataset(
        train_s,
        transform=train_transform,
        class_names=class_names,
        use_mask_roi_crop=cfg.use_mask_roi_crop,
        use_mask_geometry_channels=cfg.use_mask_geometry_channels,
        roi_margin=cfg.roi_margin,
    )
    val_ds = BUSIClassificationDataset(
        val_s,
        transform=eval_transform,
        class_names=class_names,
        use_mask_roi_crop=cfg.use_mask_roi_crop,
        use_mask_geometry_channels=cfg.use_mask_geometry_channels,
        roi_margin=cfg.roi_margin,
    )
    test_ds = BUSIClassificationDataset(
        test_s,
        transform=eval_transform,
        class_names=class_names,
        use_mask_roi_crop=cfg.use_mask_roi_crop,
        use_mask_geometry_channels=cfg.use_mask_geometry_channels,
        roi_margin=cfg.roi_margin,
    )

    sampler = make_weighted_sampler(train_s, len(class_names), cfg.seed, cfg.sampler_class_boost, cfg.train_epoch_multiplier) if cfg.use_weighted_sampler else None

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=(sampler is None),
        sampler=sampler,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=False,
    )
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

    print(f"[{display_name}] Train file counts:", Counter(y for _, y in train_s), "include_preaugmented=", cfg.use_preaugmented_train)
    print(f"[{display_name}] Val file counts  :", Counter(y for _, y in val_s), "original_only=True")
    print(f"[{display_name}] Test file counts :", Counter(y for _, y in test_s), "original_only=True")
    print(f"[{display_name}] Mask ROI crop:", cfg.use_mask_roi_crop, "roi_margin=", cfg.roi_margin)

    return train_loader, val_loader, test_loader


def make_group_splits(cfg: Config):
    groups, class_names = build_image_groups_from_root(cfg.data_root)
    train_g, val_g, test_g = stratified_group_split(groups, cfg.val_ratio, cfg.test_ratio, cfg.seed)
    print("Original image groups:", {class_names[i]: sum(int(g["label"]) == i for g in groups) for i in range(len(class_names))})
    return groups, train_g, val_g, test_g, class_names


def make_loaders(cfg: Config):
    _, train_g, val_g, test_g, class_names = make_group_splits(cfg)
    train_loader, val_loader, test_loader = make_loaders_from_group_splits(
        cfg, train_g, val_g, test_g, class_names, display_name="3-class"
    )
    return train_loader, val_loader, test_loader, class_names


# --------------------------------------------
# 7) Quick sanity checks (must run before model)
# --------------------------------------------
def sanity_check_loaders(train_loader, val_loader, test_loader, class_names):
    def count_labels(loader):
        counts = {i: 0 for i in range(len(class_names))}
        for batch in loader:
            y = batch["label"].numpy()
            for v in y:
                counts[int(v)] += 1
        return counts

    print("Classes:", class_names)
    print("Train label counts:", count_labels(train_loader))
    print("Val   label counts:", count_labels(val_loader))
    print("Test  label counts:", count_labels(test_loader))

    b = next(iter(train_loader))
    x = b["image"]
    y = b["label"]
    print("Batch image shape:", tuple(x.shape), "dtype:", x.dtype)
    print("Batch label shape:", tuple(y.shape), "dtype:", y.dtype)
    print("Example paths:", b["path"][:3])




In [ ]:
import math
import torch.nn.functional as F


# ============================================================
# 2) FiLMoS-Net: Filtered Multi-scale Morphology + Spectral Net
#    Branches: (1) Learnable Gabor bank  (2) Fixed DCT spectral
#              (3) Edge + Soft Morphology
#    Fusion: Router produces per-sample weights for branches
# ============================================================

def rgb_to_gray(x: torch.Tensor) -> torch.Tensor:
    """
    x: [B,3,H,W] -> gray: [B,1,H,W]
    """
    if x.ndim != 4 or x.size(1) != 3:
        raise ValueError(f"Expected [B,3,H,W], got {tuple(x.shape)}")
    r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
    return 0.2989 * r + 0.5870 * g + 0.1140 * b


class ConvBNAct(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 3, s: int = 1, p: Optional[int] = None):
        super().__init__()
        if p is None:
            p = k // 2
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


# ---------------------------
# Branch 1: Parametric Gabor Bank (learnable parameters -> generated kernels)
# ---------------------------
class LearnableGaborBank(nn.Module):
    """
    Generates N gabor kernels (single-channel) with learnable parameters.
    Applies on grayscale input: [B,1,H,W] -> [B,N,H,W]
    """
    def __init__(self, n_filters: int = 32, ksize: int = 21):
        super().__init__()
        if ksize % 2 == 0:
            raise ValueError("Gabor ksize must be odd.")
        self.n_filters = n_filters
        self.ksize = ksize

        # raw parameters (unconstrained) -> mapped to valid ranges in forward
        self.theta_raw = nn.Parameter(torch.randn(n_filters) * 0.3)   # orientation
        self.freq_raw  = nn.Parameter(torch.randn(n_filters) * 0.3)   # spatial frequency
        self.sigma_raw = nn.Parameter(torch.randn(n_filters) * 0.3)   # gaussian std
        self.gamma_raw = nn.Parameter(torch.randn(n_filters) * 0.3)   # aspect ratio
        self.psi_raw   = nn.Parameter(torch.randn(n_filters) * 0.3)   # phase
        self.amp_raw   = nn.Parameter(torch.zeros(n_filters))         # amplitude scaling

        # precompute coordinate grid (registered buffer)
        half = ksize // 2
        yy, xx = torch.meshgrid(
            torch.arange(-half, half + 1),
            torch.arange(-half, half + 1),
            indexing="ij",
        )
        grid = torch.stack([xx, yy], dim=0).float()  # [2,ks,ks]
        self.register_buffer("grid", grid, persistent=False)

    def _build_kernels(self, device, dtype) -> torch.Tensor:
        """
        Returns kernels: [N,1,ks,ks]
        """
        grid = self.grid.to(device=device, dtype=dtype)
        x = grid[0]  # [ks,ks]
        y = grid[1]

        # map params to ranges
        # theta in [0, pi)
        theta = torch.sigmoid(self.theta_raw) * math.pi

        # freq in [0.05, 0.45] cycles/pixel (reasonable for 224 images)
        freq = 0.05 + torch.sigmoid(self.freq_raw) * (0.45 - 0.05)

        # sigma in [2.0, 8.0]
        sigma = 2.0 + F.softplus(self.sigma_raw)

        # gamma in [0.3, 1.0]
        gamma = 0.3 + torch.sigmoid(self.gamma_raw) * 0.7

        # psi in [0, 2pi)
        psi = torch.sigmoid(self.psi_raw) * (2.0 * math.pi)

        # amplitude in [0.5, 1.5] (stable)
        amp = 0.5 + torch.sigmoid(self.amp_raw)

        kernels = []
        for i in range(self.n_filters):
            th = theta[i]
            fr = freq[i]
            sg = sigma[i]
            gm = gamma[i]
            ph = psi[i]

            # rotate coords
            x_prime = x * torch.cos(th) + y * torch.sin(th)
            y_prime = -x * torch.sin(th) + y * torch.cos(th)

            gauss = torch.exp(-0.5 * ((x_prime ** 2 + (gm ** 2) * (y_prime ** 2)) / (sg ** 2)))
            wave = torch.cos(2.0 * math.pi * fr * x_prime + ph)
            gabor = gauss * wave

            # zero-mean, unit-norm (stability)
            gabor = gabor - gabor.mean()
            gabor = gabor / (gabor.norm(p=2) + 1e-6)

            gabor = amp[i] * gabor
            kernels.append(gabor)

        k = torch.stack(kernels, dim=0)  # [N,ks,ks]
        return k.unsqueeze(1)            # [N,1,ks,ks]

    def forward(self, x_gray: torch.Tensor) -> torch.Tensor:
        if x_gray.ndim != 4 or x_gray.size(1) != 1:
            raise ValueError(f"Expected [B,1,H,W], got {tuple(x_gray.shape)}")
        kernels = self._build_kernels(device=x_gray.device, dtype=x_gray.dtype)
        # same padding
        pad = self.ksize // 2
        x = F.pad(x_gray, (pad, pad, pad, pad), mode="reflect")
        out = F.conv2d(x, kernels, bias=None, stride=1, padding=0)  # [B,N,H,W]
        return out


class GaborBranch(nn.Module):
    """
    Gray -> Gabor bank -> project to C channels -> refinement convs
    Output: [B,C,H,W]
    """
    def __init__(self, out_ch: int = 64, n_gabor: int = 32, ksize: int = 21):
        super().__init__()
        self.gabor = LearnableGaborBank(n_filters=n_gabor, ksize=ksize)
        self.proj = nn.Sequential(
            ConvBNAct(n_gabor, out_ch, k=1, s=1, p=0),
            ConvBNAct(out_ch, out_ch, k=3, s=1),
        )

    def forward(self, x_rgb: torch.Tensor) -> torch.Tensor:
        xg = rgb_to_gray(x_rgb)                # [B,1,H,W]
        f = self.gabor(xg)                     # [B,n_gabor,H,W]
        f = self.proj(f)                       # [B,C,H,W]
        return f


# ---------------------------
# Branch 2: Spectral (DCT) Branch
# - Fixed DCT 8x8 filter bank, stride=8 -> coefficient maps -> upsample
# ---------------------------
def build_dct_basis_2d(N: int = 8, normalize: bool = True) -> torch.Tensor:
    """
    Returns DCT-II basis filters: [N*N, 1, N, N]
    """
    basis = []
    for u in range(N):
        for v in range(N):
            filt = torch.zeros((N, N), dtype=torch.float32)
            for x in range(N):
                for y in range(N):
                    cu = math.sqrt(1.0 / N) if u == 0 else math.sqrt(2.0 / N)
                    cv = math.sqrt(1.0 / N) if v == 0 else math.sqrt(2.0 / N)
                    filt[x, y] = cu * cv * math.cos((math.pi * (2 * x + 1) * u) / (2 * N)) * math.cos(
                        (math.pi * (2 * y + 1) * v) / (2 * N)
                    )
            basis.append(filt)
    B = torch.stack(basis, dim=0).unsqueeze(1)  # [64,1,8,8]

    if normalize:
        # unit norm per filter
        B = B / (B.flatten(1).norm(p=2, dim=1).view(-1, 1, 1, 1) + 1e-6)
    return B


class SpectralDCTBranch(nn.Module):
    """
    Gray -> fixed DCT conv (stride 8) -> abs -> 1x1 mix -> upsample -> refine
    Output: [B,C,H,W]
    """
    def __init__(self, out_ch: int = 64, block: int = 8):
        super().__init__()
        self.block = block
        dct = build_dct_basis_2d(N=block, normalize=True)
        self.register_buffer("dct_kernels", dct, persistent=False)

        # Mix 64 coeff maps -> out_ch
        self.mix = nn.Sequential(
            ConvBNAct(block * block, out_ch, k=1, s=1, p=0),
            ConvBNAct(out_ch, out_ch, k=3, s=1),
        )

    def forward(self, x_rgb: torch.Tensor) -> torch.Tensor:
        xg = rgb_to_gray(x_rgb)  # [B,1,H,W]
        # ensure size divisible by block via reflection pad
        B, C, H, W = xg.shape
        pad_h = (self.block - (H % self.block)) % self.block
        pad_w = (self.block - (W % self.block)) % self.block
        if pad_h != 0 or pad_w != 0:
            xg = F.pad(xg, (0, pad_w, 0, pad_h), mode="reflect")
        # stride=block gives block-wise DCT maps
        coef = F.conv2d(xg, self.dct_kernels.to(xg.device, xg.dtype), stride=self.block, padding=0)  # [B,64,H/8,W/8]
        coef = coef.abs()  # magnitude-like
        feat = self.mix(coef)  # [B,out_ch,h',w']

        # upsample back to original H,W
        feat = F.interpolate(feat, size=(H, W), mode="bilinear", align_corners=False)
        return feat


# ---------------------------
# Branch 3: Edge + Soft Morphology
# - fixed edge filters + differentiable soft dilation/erosion
# ---------------------------
def fixed_edge_kernels() -> torch.Tensor:
    """
    Returns a small bank: SobelX, SobelY, Laplacian, ScharrX, ScharrY
    Shape: [5,1,3,3]
    """
    sobel_x = torch.tensor([[1, 0, -1],
                            [2, 0, -2],
                            [1, 0, -1]], dtype=torch.float32)
    sobel_y = torch.tensor([[1, 2, 1],
                            [0, 0, 0],
                            [-1, -2, -1]], dtype=torch.float32)
    lap = torch.tensor([[0, 1, 0],
                        [1, -4, 1],
                        [0, 1, 0]], dtype=torch.float32)
    scharr_x = torch.tensor([[3, 0, -3],
                             [10, 0, -10],
                             [3, 0, -3]], dtype=torch.float32)
    scharr_y = torch.tensor([[3, 10, 3],
                             [0, 0, 0],
                             [-3, -10, -3]], dtype=torch.float32)

    K = torch.stack([sobel_x, sobel_y, lap, scharr_x, scharr_y], dim=0).unsqueeze(1)
    # normalize per kernel
    K = K / (K.flatten(1).norm(p=2, dim=1).view(-1, 1, 1, 1) + 1e-6)
    return K


def soft_dilate(x: torch.Tensor, k: int = 3, beta: float = 10.0) -> torch.Tensor:
    """
    Differentiable approximation of dilation using log-sum-exp over local window.
    """
    pad = k // 2
    xpad = F.pad(x, (pad, pad, pad, pad), mode="reflect")
    # unfold: [B,C, H*W, k*k]
    patches = xpad.unfold(2, k, 1).unfold(3, k, 1)  # [B,C,H,W,k,k]
    patches = patches.contiguous().view(*patches.shape[:4], -1)  # [B,C,H,W,k*k]
    # logsumexp over window
    return (1.0 / beta) * torch.logsumexp(beta * patches, dim=-1)


def soft_erode(x: torch.Tensor, k: int = 3, beta: float = 10.0) -> torch.Tensor:
    return -soft_dilate(-x, k=k, beta=beta)


class EdgeMorphBranch(nn.Module):
    """
    Gray -> fixed edge bank -> combine -> soft morphology -> project to C
    Output: [B,C,H,W]
    """
    def __init__(self, out_ch: int = 64, beta: float = 10.0):
        super().__init__()
        K = fixed_edge_kernels()
        self.register_buffer("edge_kernels", K, persistent=False)
        self.beta = beta

        # edge bank -> 5 channels, then to out_ch
        self.proj = nn.Sequential(
            ConvBNAct(5 * 3, out_ch, k=1, s=1, p=0),  # (edge + dilate + erode) concatenated
            ConvBNAct(out_ch, out_ch, k=3, s=1),
        )

    def forward(self, x_rgb: torch.Tensor) -> torch.Tensor:
        xg = rgb_to_gray(x_rgb)  # [B,1,H,W]
        pad = 1
        xpad = F.pad(xg, (pad, pad, pad, pad), mode="reflect")
        edges = F.conv2d(xpad, self.edge_kernels.to(xg.device, xg.dtype), stride=1, padding=0)  # [B,5,H,W]
        edges = torch.tanh(edges)  # stabilize

        dil = soft_dilate(edges, k=3, beta=self.beta)
        ero = soft_erode(edges, k=3, beta=self.beta)

        feat = torch.cat([edges, dil, ero], dim=1)  # [B,15,H,W]
        feat = self.proj(feat)                     # [B,C,H,W]
        return feat


# ---------------------------
# Router + Fusion
# ---------------------------
class BranchRouter(nn.Module):
    """
    Takes pooled features from each branch and outputs soft weights per sample.
    """
    def __init__(self, ch: int, hidden: int = 128, n_branches: int = 3):
        super().__init__()
        self.n_branches = n_branches
        self.mlp = nn.Sequential(
            nn.Linear(ch * n_branches, hidden),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(hidden, n_branches),
        )

    def forward(self, feats: List[torch.Tensor]) -> torch.Tensor:
        # feats list each: [B,C,H,W]
        pooled = [F.adaptive_avg_pool2d(f, 1).flatten(1) for f in feats]  # each [B,C]
        x = torch.cat(pooled, dim=1)                                      # [B,3C]
        w = self.mlp(x)                                                   # [B,3]
        w = F.softmax(w, dim=1)
        return w


class FiLMoSNet(nn.Module):
    def __init__(
        self,
        num_classes: int = 3,
        base_ch: int = 64,
        gabor_filters: int = 32,
        gabor_ksize: int = 21,
        dct_block: int = 8,
        morph_beta: float = 10.0,
    ):
        super().__init__()

        self.branch_gabor = GaborBranch(out_ch=base_ch, n_gabor=gabor_filters, ksize=gabor_ksize)
        self.branch_spec  = SpectralDCTBranch(out_ch=base_ch, block=dct_block)
        self.branch_morph = EdgeMorphBranch(out_ch=base_ch, beta=morph_beta)

        self.router = BranchRouter(ch=base_ch, hidden=128, n_branches=3)

        # post-fusion refinement (small backbone)
        self.refine = nn.Sequential(
            ConvBNAct(base_ch, base_ch, k=3, s=1),
            ConvBNAct(base_ch, base_ch * 2, k=3, s=2),  # downsample
            ConvBNAct(base_ch * 2, base_ch * 2, k=3, s=1),
            ConvBNAct(base_ch * 2, base_ch * 4, k=3, s=2),  # downsample
            ConvBNAct(base_ch * 4, base_ch * 4, k=3, s=1),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(1),
            nn.Dropout(p=0.2),
            nn.Linear(base_ch * 4, num_classes),
        )

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Returns dict for transparency:
          - logits: [B,3]
          - router_w: [B,3]
        """
        f1 = self.branch_gabor(x)
        f2 = self.branch_spec(x)
        f3 = self.branch_morph(x)

        w = self.router([f1, f2, f3])  # [B,3]
        # weighted fusion: sum_i w_i * f_i
        # reshape weights for broadcasting
        w1 = w[:, 0].view(-1, 1, 1, 1)
        w2 = w[:, 1].view(-1, 1, 1, 1)
        w3 = w[:, 2].view(-1, 1, 1, 1)
        fused = w1 * f1 + w2 * f2 + w3 * f3  # [B,C,H,W]

        z = self.refine(fused)
        logits = self.head(z)

        return {"logits": logits, "router_w": w}

# ============================================================
# Strong transfer-learning baseline for small BUSI datasets
# ============================================================
class PretrainedClassifier(nn.Module):
    def __init__(self, model_name: str = "convnext_tiny", num_classes: int = 3, pretrained: bool = True):
        super().__init__()
        self.model_name = model_name.lower()
        self.num_classes = num_classes

        if self.model_name == "convnext_tiny":
            weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = models.convnext_tiny(weights=weights)
            in_features = self.backbone.classifier[-1].in_features
            self.backbone.classifier[-1] = nn.Linear(in_features, num_classes)
        elif self.model_name == "convnext_small":
            weights = models.ConvNeXt_Small_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = models.convnext_small(weights=weights)
            in_features = self.backbone.classifier[-1].in_features
            self.backbone.classifier[-1] = nn.Linear(in_features, num_classes)
        elif self.model_name == "efficientnet_b0":
            weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = models.efficientnet_b0(weights=weights)
            in_features = self.backbone.classifier[-1].in_features
            self.backbone.classifier[-1] = nn.Linear(in_features, num_classes)
        else:
            raise ValueError("model_name must be 'convnext_tiny', 'convnext_small', or 'efficientnet_b0'")

    def set_backbone_trainable(self, trainable: bool) -> None:
        for p in self.backbone.parameters():
            p.requires_grad = trainable

        # Always keep the classification head trainable.
        head = self.backbone.classifier
        for p in head.parameters():
            p.requires_grad = True

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        logits = self.backbone(x)
        router_w = torch.zeros((x.size(0), 3), device=x.device, dtype=logits.dtype)
        return {"logits": logits, "router_w": router_w}


def build_model(cfg: Config, num_classes: int) -> nn.Module:
    if cfg.model_name.lower() == "filmos":
        return FiLMoSNet(
            num_classes=num_classes,
            base_ch=64,
            gabor_filters=32,
            gabor_ksize=21,
            dct_block=8,
            morph_beta=10.0,
        )

    return PretrainedClassifier(
        model_name=cfg.model_name,
        num_classes=num_classes,
        pretrained=cfg.pretrained,
    )



In [ ]:
# ------------------------------------------------------------
# Sanity check for model forward (must run before training)
# ------------------------------------------------------------
@torch.no_grad()
def sanity_check_model_forward(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    batch = next(iter(loader))
    x = batch["image"].to(device)
    y = batch["label"].to(device)

    out = model(x)
    logits = out["logits"]
    w = out["router_w"]

    print("Forward OK.")
    print("Input:", tuple(x.shape), x.dtype, x.device)
    print("Logits:", tuple(logits.shape), logits.dtype)
    print("Router weights:", tuple(w.shape))
    print("Router weights sample (first 3 rows):\n", w[:3].cpu())

    # check logits match classes
    if logits.size(1) != 3:
        raise RuntimeError("Logits second dim must be 3 for benign/malignant/normal.")
    # compute quick loss numeric check
    loss = F.cross_entropy(logits, y)
    print("CE loss (single batch):", float(loss.cpu()))

In [ ]:
def compute_class_weights_from_counts(counts: Dict[int, int], num_classes: int, power: float = 1.0) -> torch.Tensor:
    """
    Inverse-frequency class weights normalized around 1.0.
    power=1.0 is full inverse frequency; power=0.5 is a safer sqrt correction.
    """
    total = sum(counts.values())
    raw = []
    for c in range(num_classes):
        if counts.get(c, 0) == 0:
            raise ValueError(f"class {c} has 0 samples in training counts.")
        raw.append((total / (num_classes * counts[c])) ** power)
    w = torch.tensor(raw, dtype=torch.float32)
    return w / w.mean()


def compute_class_weights_from_loader(loader: DataLoader, num_classes: int, power: float = 1.0) -> torch.Tensor:
    counts = Counter()
    for _, y in loader.dataset.samples:
        counts[int(y)] += 1
    return compute_class_weights_from_counts(dict(counts), num_classes=num_classes, power=power)


def confusion_matrix_torch(y_true: torch.Tensor, y_pred: torch.Tensor, num_classes: int) -> torch.Tensor:
    """
    y_true, y_pred: [N] on CPU or GPU
    returns CM [C,C] where rows=true, cols=pred
    """
    if y_true.ndim != 1 or y_pred.ndim != 1:
        raise ValueError("y_true and y_pred must be 1D")
    cm = torch.zeros((num_classes, num_classes), dtype=torch.long, device=y_true.device)
    for t, p in zip(y_true, y_pred):
        cm[t.long(), p.long()] += 1
    return cm


def metrics_from_confusion(cm: torch.Tensor) -> Dict[str, object]:
    """
    cm: [C,C] on CPU
    Computes per-class sensitivity/recall, specificity, precision, f1; macro-f1, accuracy.
    """
    cm = cm.to(torch.float64)
    C = cm.size(0)
    total = cm.sum()
    correct = torch.diag(cm).sum()
    acc = (correct / (total + 1e-12)).item()

    per_class = []
    for c in range(C):
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = total - tp - fn - fp

        recall = (tp / (tp + fn + 1e-12)).item()
        spec = (tn / (tn + fp + 1e-12)).item()
        prec = (tp / (tp + fp + 1e-12)).item()
        f1 = (2 * tp / (2 * tp + fp + fn + 1e-12)).item()

        per_class.append({
            "recall_sens": recall,
            "specificity": spec,
            "precision": prec,
            "f1": f1,
            "support": int(cm[c, :].sum().item()),
        })

    macro_f1 = float(np.mean([d["f1"] for d in per_class]))
    macro_recall = float(np.mean([d["recall_sens"] for d in per_class]))
    macro_spec = float(np.mean([d["specificity"] for d in per_class]))
    min_recall = float(min(d["recall_sens"] for d in per_class))
    min_f1 = float(min(d["f1"] for d in per_class))

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "macro_recall": macro_recall,
        "macro_specificity": macro_spec,
        "min_recall": min_recall,
        "min_f1": min_f1,
        "per_class": per_class,
    }


def try_compute_auc_ovr(y_true_np: np.ndarray, prob_np: np.ndarray, num_classes: int) -> Optional[float]:
    try:
        from sklearn.metrics import roc_auc_score
    except Exception:
        return None

    try:
        auc = roc_auc_score(y_true_np, prob_np, multi_class="ovr", average="macro")
        return float(auc)
    except Exception:
        return None


def metrics_from_logits(
    logits: torch.Tensor,
    y_true: torch.Tensor,
    num_classes: int,
    logit_bias: Optional[torch.Tensor] = None,
    loss: Optional[float] = None,
) -> Dict[str, object]:
    logits = logits.detach().float().cpu()
    y_true = y_true.detach().cpu()
    if logit_bias is not None:
        logits = logits + logit_bias.detach().float().cpu().view(1, -1)

    probs = torch.softmax(logits, dim=1).numpy()
    y_np = y_true.numpy().astype(np.int64)
    y_pred = torch.argmax(logits, dim=1)
    cm = confusion_matrix_torch(y_true, y_pred, num_classes=num_classes).cpu()
    m = metrics_from_confusion(cm)
    auc = try_compute_auc_ovr(y_np, probs, num_classes=num_classes)

    return {
        "loss": loss,
        "cm": cm,
        "auc_ovr_macro": auc,
        "logit_bias": logit_bias.detach().float().cpu() if logit_bias is not None else None,
        **m,
    }


def balanced_selection_score(metrics: Dict[str, object]) -> float:
    # Keep calibration and checkpoint selection aligned with the reported primary metric.
    return float(metrics["macro_f1"])


def tune_logit_bias(
    logits: torch.Tensor,
    y_true: torch.Tensor,
    num_classes: int,
    search_min: float = -2.0,
    search_max: float = 2.0,
    steps: int = 41,
) -> Tuple[torch.Tensor, Dict[str, object]]:
    """
    Validation-only decision calibration. Class 0 bias is fixed at 0 to remove redundancy.
    This is useful when AUC is high but argmax accuracy/F1 are poor.
    """
    grid = torch.linspace(search_min, search_max, steps)
    best_bias = torch.zeros(num_classes)
    best_metrics = metrics_from_logits(logits, y_true, num_classes, best_bias)
    best_score = balanced_selection_score(best_metrics)

    if num_classes != 3:
        return best_bias, best_metrics

    for b1 in grid:
        for b2 in grid:
            bias = torch.tensor([0.0, float(b1), float(b2)])
            m = metrics_from_logits(logits, y_true, num_classes, bias)
            score = balanced_selection_score(m)
            if score > best_score + 1e-12 or (abs(score - best_score) <= 1e-12 and m["accuracy"] > best_metrics["accuracy"]):
                best_score = score
                best_bias = bias
                best_metrics = m

    return best_bias, best_metrics



In [ ]:
@torch.no_grad()
def collect_logits(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    use_tta: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor, float]:
    model.eval()

    all_logits = []
    all_y = []
    total_loss = 0.0
    total_n = 0

    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)

        out = model(x)
        logits = out["logits"]
        if use_tta:
            tta_logits = [logits]
            tta_logits.append(model(torch.flip(x, dims=[3]))["logits"])
            tta_logits.append(model(torch.flip(x, dims=[2]))["logits"])
            x_bright = torch.clamp(x * 1.03, -3.0, 3.0)
            x_dark = torch.clamp(x * 0.97, -3.0, 3.0)
            tta_logits.append(model(x_bright)["logits"])
            tta_logits.append(model(x_dark)["logits"])
            logits = torch.stack(tta_logits, dim=0).mean(dim=0)

        loss = F.cross_entropy(logits, y)
        bs = x.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs

        all_logits.append(logits.detach().float().cpu())
        all_y.append(y.detach().cpu())

    return torch.cat(all_logits, dim=0), torch.cat(all_y, dim=0), total_loss / max(total_n, 1)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    num_classes: int,
    logit_bias: Optional[torch.Tensor] = None,
    use_tta: bool = False,
) -> Dict[str, object]:
    logits, y_true, loss = collect_logits(model, loader, device, use_tta=use_tta)
    return metrics_from_logits(logits, y_true, num_classes=num_classes, logit_bias=logit_bias, loss=loss)



In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    scaler: Optional[torch.cuda.amp.GradScaler],
    class_weights: Optional[torch.Tensor],
    label_smoothing: float = 0.0,
    focal_gamma: float = 0.0,
    mixup_alpha: float = 0.0,
    mixup_prob: float = 0.0,
    max_grad_norm: float = 1.0,
) -> Dict[str, float]:
    model.train()

    total_loss = 0.0
    total_n = 0
    correct = 0

    if class_weights is not None:
        class_weights = class_weights.to(device)

    def criterion(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(
            logits,
            y,
            weight=class_weights,
            label_smoothing=label_smoothing,
            reduction="none",
        )
        if focal_gamma <= 0:
            return ce.mean()
        pt = torch.softmax(logits.detach(), dim=1).gather(1, y.view(-1, 1)).squeeze(1).clamp(1e-4, 1.0)
        focal = (1.0 - pt).pow(float(focal_gamma))
        return (focal * ce).mean()

    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        use_mixup = mixup_alpha > 0 and x.size(0) > 1 and random.random() < mixup_prob
        if use_mixup:
            lam = float(np.random.beta(mixup_alpha, mixup_alpha))
            lam = max(lam, 1.0 - lam)
            index = torch.randperm(x.size(0), device=device)
            x_in = lam * x + (1.0 - lam) * x[index]
            y_mix = y[index]
        else:
            lam = 1.0
            x_in = x
            y_mix = y

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.cuda.amp.autocast():
                out = model(x_in)
                logits = out["logits"]
                if use_mixup:
                    loss = lam * criterion(logits, y) + (1.0 - lam) * criterion(logits, y_mix)
                else:
                    loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(x_in)
            logits = out["logits"]
            if use_mixup:
                loss = lam * criterion(logits, y) + (1.0 - lam) * criterion(logits, y_mix)
            else:
                loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        bs = x.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs
        pred = torch.argmax(logits.detach(), dim=1)
        correct += int((pred == y).sum().item())

    return {
        "train_loss": total_loss / max(total_n, 1),
        "train_acc": correct / max(total_n, 1),
    }


def save_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    best_score: float,
    logit_bias: Optional[torch.Tensor] = None,
):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "best_score": best_score,
            "logit_bias": logit_bias.detach().float().cpu() if logit_bias is not None else None,
        },
        str(path),
    )


def format_metrics(prefix: str, m: Dict[str, object]) -> None:
    print(f"  {prefix}: loss={m['loss']:.4f} acc={m['accuracy']:.4f} macro_f1={m['macro_f1']:.4f} "
          f"macro_recall={m['macro_recall']:.4f} min_recall={m['min_recall']:.4f} "
          f"macro_spec={m['macro_specificity']:.4f} "
          f"auc={m['auc_ovr_macro'] if m['auc_ovr_macro'] is not None else 'NA'}")


def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    num_classes: int,
    class_weights: torch.Tensor,
    epochs: int,
    lr: float,
    weight_decay: float,
    use_amp: bool,
    label_smoothing: float = 0.0,
    focal_gamma: float = 0.0,
    mixup_alpha: float = 0.0,
    mixup_prob: float = 0.0,
    min_lr: float = 1e-6,
    patience: int = 14,
    freeze_epochs: int = 0,
    ckpt_dir: str = "./checkpoints_filmOS",
    monitor: str = "balanced",
):
    ckpt_dir = Path(ckpt_dir)

    if freeze_epochs > 0 and hasattr(model, "set_backbone_trainable"):
        model.set_backbone_trainable(False)
        print(f"Freezing pretrained backbone for {freeze_epochs} warmup epochs.")

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs), eta_min=min_lr)

    scaler = torch.cuda.amp.GradScaler() if (use_amp and device.type == "cuda") else None

    best_score = -1e9
    best_epoch = -1
    best_bias = torch.zeros(num_classes)
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        if freeze_epochs > 0 and epoch == freeze_epochs + 1 and hasattr(model, "set_backbone_trainable"):
            model.set_backbone_trainable(True)
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr * 0.5, weight_decay=weight_decay)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs - freeze_epochs), eta_min=min_lr)
            print("Unfroze pretrained backbone for full fine-tuning.")

        tr = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            class_weights=class_weights,
            label_smoothing=label_smoothing,
            focal_gamma=focal_gamma,
            mixup_alpha=mixup_alpha,
            mixup_prob=mixup_prob,
            max_grad_norm=1.0,
        )

        val_logits, val_y, val_loss = collect_logits(model, val_loader, device, use_tta=False)
        raw_val = metrics_from_logits(val_logits, val_y, num_classes=num_classes, loss=val_loss)
        epoch_bias, cal_val = tune_logit_bias(val_logits, val_y, num_classes=num_classes)
        cal_val["loss"] = val_loss

        if monitor == "macro_f1":
            score = cal_val["macro_f1"]
        elif monitor == "balanced":
            score = balanced_selection_score(cal_val)
        elif monitor == "auc_ovr_macro":
            score = raw_val["auc_ovr_macro"] if raw_val["auc_ovr_macro"] is not None else -1e9
        else:
            raise ValueError("monitor must be 'macro_f1', 'balanced', or 'auc_ovr_macro'")

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"\nEpoch {epoch}/{epochs} lr={current_lr:.2e}")
        overfit_gap = tr["train_acc"] - cal_val["accuracy"]
        print(f"  Train: loss={tr['train_loss']:.4f} acc={tr['train_acc']:.4f} overfit_gap={overfit_gap:.4f}")
        if overfit_gap > 0.08:
            print("  Overfit warning: train accuracy is much higher than calibrated validation accuracy.")
        format_metrics("Val raw", raw_val)
        format_metrics("Val calibrated", cal_val)
        print("  Val logit bias:", [round(float(v), 3) for v in epoch_bias.tolist()])

        for i, pc in enumerate(cal_val["per_class"]):
            print(f"    Class {i}: recall={pc['recall_sens']:.4f} spec={pc['specificity']:.4f} "
                  f"prec={pc['precision']:.4f} f1={pc['f1']:.4f} support={pc['support']}")

        if score > best_score + 1e-6:
            best_score = score
            best_epoch = epoch
            best_bias = epoch_bias.clone()
            bad_epochs = 0
            save_checkpoint(ckpt_dir / "best.pt", model, optimizer, epoch, best_score, logit_bias=best_bias)
            print(f"  Saved new best checkpoint (epoch={epoch}, {monitor}={best_score:.4f})")
        else:
            bad_epochs += 1

        save_checkpoint(ckpt_dir / "last.pt", model, optimizer, epoch, best_score, logit_bias=best_bias)
        scheduler.step()

        if bad_epochs >= patience:
            print(f"\nEarly stopping: no {monitor} improvement for {patience} epochs.")
            break

    print(f"\nTraining done. Best epoch={best_epoch} best_{monitor}={best_score:.4f}")
    print("Best validation logit bias:", [round(float(v), 3) for v in best_bias.tolist()])
    return ckpt_dir / "best.pt"



In [ ]:
def train_eval_one_run(
    base_cfg: Config,
    model_name: str,
    run_seed: int,
    device: torch.device,
    train_g: List[Dict[str, object]],
    val_g: List[Dict[str, object]],
    test_g: List[Dict[str, object]],
    class_names: List[str],
):
    seed_everything(run_seed)
    cfg = replace(base_cfg, model_name=model_name)

    train_loader, val_loader, test_loader = make_loaders_from_group_splits(
        cfg, train_g, val_g, test_g, class_names, display_name=f"3-class/{model_name}/seed{run_seed}"
    )
    class_w = compute_class_weights_from_loader(train_loader, num_classes=3, power=cfg.class_weight_power)
    print("Class weights:", class_w.tolist())

    model = build_model(cfg, num_classes=3).to(device)
    print("Model:", cfg.model_name, "pretrained=", cfg.pretrained, "run_seed=", run_seed)

    best_ckpt = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        num_classes=3,
        class_weights=class_w,
        epochs=cfg.epochs,
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
        use_amp=True,
        label_smoothing=cfg.label_smoothing,
        focal_gamma=cfg.focal_gamma,
        mixup_alpha=cfg.mixup_alpha,
        mixup_prob=cfg.mixup_prob,
        min_lr=cfg.min_lr,
        patience=cfg.patience,
        freeze_epochs=cfg.freeze_epochs,
        ckpt_dir=f"./checkpoints_stage1_{cfg.model_name}_seed{run_seed}",
        monitor="macro_f1",
    )

    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    logit_bias = ckpt.get("logit_bias")
    if logit_bias is None:
        logit_bias = torch.zeros(3)

    val_logits, val_y, val_loss = collect_logits(model, val_loader, device, use_tta=True)
    test_logits, test_y, test_loss = collect_logits(model, test_loader, device, use_tta=True)

    # TTA changes the logits, so calibrate once on TTA validation logits and reuse on test.
    logit_bias, val_m = tune_logit_bias(val_logits, val_y, num_classes=3)
    val_m["loss"] = val_loss
    test_m = metrics_from_logits(test_logits, test_y, num_classes=3, logit_bias=logit_bias, loss=test_loss)
    print("\nSingle-run calibrated TTA validation:")
    format_metrics("Val", val_m)
    print("Single-run calibrated TTA test:")
    format_metrics("Test", test_m)

    return {
        "cfg": cfg,
        "class_names": class_names,
        "val_logits": val_logits,
        "val_y": val_y,
        "val_loss": val_loss,
        "test_logits": test_logits,
        "test_y": test_y,
        "test_loss": test_loss,
        "bias": logit_bias.detach().cpu(),
    }


@torch.no_grad()
def collect_inference_logits(model: nn.Module, loader: DataLoader, device: torch.device, use_tta: bool = True):
    model.eval()
    all_logits = []
    all_y = []
    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["label"]
        out = model(x)
        logits = out["logits"]
        if use_tta:
            tta_logits = [logits]
            tta_logits.append(model(torch.flip(x, dims=[3]))["logits"])
            tta_logits.append(model(torch.flip(x, dims=[2]))["logits"])
            tta_logits.append(model(torch.clamp(x * 1.03, -3.0, 3.0))["logits"])
            tta_logits.append(model(torch.clamp(x * 0.97, -3.0, 3.0))["logits"])
            logits = torch.stack(tta_logits, dim=0).mean(dim=0)
        all_logits.append(logits.detach().float().cpu())
        all_y.append(y.detach().cpu())
    return torch.cat(all_logits, dim=0), torch.cat(all_y, dim=0)


def train_binary_specialist(
    base_cfg: Config,
    model_name: str,
    run_seed: int,
    device: torch.device,
    train_g: List[Dict[str, object]],
    val_g: List[Dict[str, object]],
    test_g: List[Dict[str, object]],
):
    seed_everything(run_seed)
    cfg = replace(
        base_cfg,
        model_name=model_name,
        epochs=max(base_cfg.epochs, 55),
        freeze_epochs=max(base_cfg.freeze_epochs, 8),
        lr=min(base_cfg.lr, 4e-5),
        label_smoothing=0.03,
        class_weight_power=max(base_cfg.class_weight_power, 0.45),
        patience=max(base_cfg.patience, 14),
    )
    binary_names = ["benign", "malignant"]
    train_loader, val_loader, test_loader = make_loaders_from_group_splits(
        cfg,
        train_g,
        val_g,
        test_g,
        binary_names,
        allowed_labels={0, 1},
        relabel_map={0: 0, 1: 1},
        display_name=f"benign-vs-malignant/{model_name}/seed{run_seed}",
    )
    class_w = compute_class_weights_from_loader(train_loader, num_classes=2, power=cfg.class_weight_power)
    print("Binary specialist class weights:", class_w.tolist())

    model = build_model(cfg, num_classes=2).to(device)
    best_ckpt = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        num_classes=2,
        class_weights=class_w,
        epochs=cfg.epochs,
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
        use_amp=True,
        label_smoothing=cfg.label_smoothing,
        focal_gamma=cfg.focal_gamma,
        mixup_alpha=cfg.mixup_alpha,
        mixup_prob=cfg.mixup_prob,
        min_lr=cfg.min_lr,
        patience=cfg.patience,
        freeze_epochs=cfg.freeze_epochs,
        ckpt_dir=f"./checkpoints_stage2_bm_{cfg.model_name}_seed{run_seed}",
        monitor="balanced",
    )

    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    val_logits, val_y, val_loss = collect_logits(model, val_loader, device, use_tta=True)
    test_logits, test_y, test_loss = collect_logits(model, test_loader, device, use_tta=True)
    bias = ckpt.get("logit_bias")
    if bias is None:
        bias = torch.zeros(2)

    val_m = metrics_from_logits(val_logits, val_y, num_classes=2, logit_bias=bias, loss=val_loss)
    test_m = metrics_from_logits(test_logits, test_y, num_classes=2, logit_bias=bias, loss=test_loss)
    print("\nBinary specialist calibrated TTA validation:")
    format_metrics("Val B/M", val_m)
    print("Binary specialist calibrated TTA test:")
    format_metrics("Test B/M", test_m)

    # Run the binary specialist on the full 3-class validation/test order.
    # This gives benign/malignant logits for every sample, while normal is still decided by stage 1.
    _, full_val_loader, full_test_loader = make_loaders_from_group_splits(
        cfg, train_g, val_g, test_g, ["benign", "malignant", "normal"], display_name="stage2/full-order-inference"
    )
    full_val_logits, full_val_y = collect_inference_logits(model, full_val_loader, device, use_tta=True)
    full_test_logits, full_test_y = collect_inference_logits(model, full_test_loader, device, use_tta=True)

    return {
        "val_logits": full_val_logits,
        "val_y": full_val_y,
        "val_loss": val_loss,
        "test_logits": full_test_logits,
        "test_y": full_test_y,
        "test_loss": test_loss,
        "bias": bias.detach().cpu(),
    }


def combine_hierarchical_logits(
    stage1_logits: torch.Tensor,
    bm_logits: torch.Tensor,
    stage1_bias: torch.Tensor,
    bm_bias: torch.Tensor,
    normal_logit_shift: float = 0.0,
    bm_strength: float = 1.0,
) -> torch.Tensor:
    s1 = stage1_logits.detach().float().cpu() + stage1_bias.detach().float().cpu().view(1, -1)
    bm = bm_logits.detach().float().cpu() + bm_bias.detach().float().cpu().view(1, -1)
    combined = s1.clone()
    combined[:, 0:2] = (1.0 - bm_strength) * combined[:, 0:2] + bm_strength * bm
    combined[:, 2] = combined[:, 2] + float(normal_logit_shift)
    return combined


def tune_hierarchical_decision(
    stage1_val_logits: torch.Tensor,
    bm_val_logits: torch.Tensor,
    val_y: torch.Tensor,
    stage1_bias: torch.Tensor,
    bm_bias: torch.Tensor,
):
    best_params = {"normal_shift": 0.0, "bm_strength": 1.0}
    best_logits = combine_hierarchical_logits(stage1_val_logits, bm_val_logits, stage1_bias, bm_bias)
    best_metrics = metrics_from_logits(best_logits, val_y, num_classes=3)
    best_score = balanced_selection_score(best_metrics)

    for normal_shift in torch.linspace(-1.0, 1.0, 21):
        for bm_strength in torch.linspace(0.65, 1.35, 15):
            logits = combine_hierarchical_logits(
                stage1_val_logits,
                bm_val_logits,
                val_y.new_tensor(stage1_bias).float(),
                val_y.new_tensor(bm_bias).float(),
                normal_logit_shift=float(normal_shift),
                bm_strength=float(bm_strength),
            )
            m = metrics_from_logits(logits, val_y, num_classes=3)
            score = balanced_selection_score(m)
            if score > best_score + 1e-12 or (abs(score - best_score) <= 1e-12 and m["accuracy"] > best_metrics["accuracy"]):
                best_score = score
                best_metrics = m
                best_params = {"normal_shift": float(normal_shift), "bm_strength": float(bm_strength)}
                best_logits = logits

    return best_params, best_metrics, best_logits


def evaluate_ensemble(run_outputs: List[Dict[str, object]], class_names: List[str], binary_output: Optional[Dict[str, object]] = None):
    val_y = run_outputs[0]["val_y"]
    test_y = run_outputs[0]["test_y"]
    # Average raw member logits, then calibrate the ensemble exactly once on validation.
    val_logits = torch.stack([r["val_logits"] for r in run_outputs], dim=0).mean(dim=0)
    test_logits = torch.stack([r["test_logits"] for r in run_outputs], dim=0).mean(dim=0)
    val_loss = float(np.mean([r["val_loss"] for r in run_outputs]))
    test_loss = float(np.mean([r["test_loss"] for r in run_outputs]))

    ens_bias, ens_val = tune_logit_bias(val_logits, val_y, num_classes=3, search_min=-2.0, search_max=2.0, steps=41)
    ens_val["loss"] = val_loss
    ens_test = metrics_from_logits(test_logits, test_y, num_classes=3, logit_bias=ens_bias, loss=test_loss)

    print("\n==== STAGE-1 ENSEMBLE VALIDATION RESULTS ====")
    format_metrics("Val ensemble", ens_val)
    print("Ensemble logit bias:", [round(float(v), 3) for v in ens_bias.tolist()])
    print("Validation confusion matrix (rows=true, cols=pred):\n", ens_val["cm"].numpy())

    print("\n==== STAGE-1 ENSEMBLE TEST RESULTS ====")
    format_metrics("Test ensemble", ens_test)
    print("Test confusion matrix (rows=true, cols=pred):\n", ens_test["cm"].numpy())

    final_test = ens_test
    if binary_output is not None:
        params, hier_val, _ = tune_hierarchical_decision(
            val_logits,
            binary_output["val_logits"],
            val_y,
            ens_bias,
            binary_output["bias"],
        )
        hier_val["loss"] = val_loss
        hier_test_logits = combine_hierarchical_logits(
            test_logits,
            binary_output["test_logits"],
            ens_bias,
            binary_output["bias"],
            normal_logit_shift=params["normal_shift"],
            bm_strength=params["bm_strength"],
        )
        hier_test = metrics_from_logits(hier_test_logits, test_y, num_classes=3, loss=test_loss)
        final_test = hier_test

        print("\n==== HIERARCHICAL VALIDATION RESULTS ====")
        format_metrics("Val hierarchical", hier_val)
        print("Hierarchical decision params:", {k: round(v, 3) for k, v in params.items()})
        print("Validation confusion matrix (rows=true, cols=pred):\n", hier_val["cm"].numpy())

        print("\n==== HIERARCHICAL TEST RESULTS ====")
        format_metrics("Test hierarchical", hier_test)
        print("Test confusion matrix (rows=true, cols=pred):\n", hier_test["cm"].numpy())

    for i, pc in enumerate(final_test["per_class"]):
        print(f"Class {class_names[i]}: recall={pc['recall_sens']:.4f} spec={pc['specificity']:.4f} "
              f"prec={pc['precision']:.4f} f1={pc['f1']:.4f} support={pc['support']}")

    if final_test["accuracy"] < 0.91 or final_test["macro_f1"] < 0.91 or final_test["macro_recall"] < 0.91:
        print("\n91% target not reached on this fixed split. For a paper-grade estimate, run stratified group cross-validation or add external ultrasound data.")
    else:
        print("\nTarget reached on this fixed split with the hierarchical decision system.")

    return ens_val, final_test


def main():
    local_data_root = "Dataset_BUSI_with_GT"
    colab_data_root = "/content/Dataset_BUSI_with_GT/dataset/Dataset_BUSI_with_GT/"
    data_root = local_data_root if os.path.isdir(local_data_root) else colab_data_root
    cfg = Config(
        data_root=data_root,
        img_size=128,
        batch_size=32,
        num_workers=2,
        seed=42,
        val_ratio=0.15,
        test_ratio=0.15,

        use_preaugmented_train=False,
        use_online_train_augmentation=True,
        use_weighted_sampler=False,
        train_epoch_multiplier=1.0,
        use_mask_roi_crop=False,
        use_mask_geometry_channels=True,
        roi_margin=0.15,
        apply_basic_transforms=True,

        pretrained=False,
        epochs=30,
        freeze_epochs=0,
        lr=7e-4,
        min_lr=1e-6,
        weight_decay=1e-4,
        label_smoothing=0.02,
        class_weight_power=0.65,
        focal_gamma=0.0,
        mixup_alpha=0.0,
        mixup_prob=0.0,
        sampler_class_boost=None,
        patience=8,
    )

    seed_everything(cfg.seed)
    _, train_g, val_g, test_g, class_names = make_group_splits(cfg)
    train_loader, val_loader, test_loader = make_loaders_from_group_splits(
        cfg, train_g, val_g, test_g, class_names, display_name="sanity/3-class"
    )
    sanity_check_loaders(train_loader, val_loader, test_loader, class_names)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # Final FiLMoS strategy: architecture unchanged; mask geometry is preserved in preprocessing.
    ensemble_plan = [
        ("filmos", 42),
    ]

    run_outputs = []
    for model_name, run_seed in ensemble_plan:
        print("\n" + "=" * 80)
        print(f"Starting stage-1 ensemble member: model={model_name}, run_seed={run_seed}")
        print("=" * 80)
        run_outputs.append(train_eval_one_run(cfg, model_name, run_seed, device, train_g, val_g, test_g, class_names))

    evaluate_ensemble(run_outputs, class_names, binary_output=None)


if __name__ == "__main__":
    main()
